# AS cluster ranking across samples

Этот ноутбук собирает в одном месте старую rheum-логику:
- AS-клоны размечаются как в старом ноутбуке через `threshold=1` по списку `as_seqs`.
- Volcano строятся по каждому образцу с отдельным цветом для кластеров, где есть `as_pattern`.
- Кластеры между образцами можно объединять по общему CDR3 или по одной замене: `MERGE_SUBSTITUTIONS=0/1`.
- После merge считаются summary-метрики, Fisher p-value и TOPSIS-рэнкинг.
- Для top-кластеров рисуются logo-плоты с подписью по V/J генам.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from as_cluster_ranking_helpers import (
    DEFAULT_AS_SEQS,
    annotate_run_volcano_summary,
    build_reference_matcher,
    load_many_runs,
    merge_clusters_by_cdr3_threshold,
    plot_ranked_metrics,
    plot_sample_volcano_grid,
    plot_top_cluster_logos,
    prepare_merged_clonotypes,
    rank_clusters_with_topsis,
    summarize_merged_clusters,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)


In [ ]:
RUNS_ROOT = Path("/projects/immunestatus/rheum/redcea/runs")
SAMPLE_NAMES = sorted([
    path.name
    for path in RUNS_ROOT.iterdir()
    if path.is_dir() and path.name.startswith("as_")
])

# Старый AS reference из 9_first_attempt_to_find_truth.ipynb
AS_SEQS = DEFAULT_AS_SEQS
AS_MATCH_THRESHOLD = 1

# 0 = старый strict merge по общему CDR3, 1 = merge по одной замене
MERGE_SUBSTITUTIONS = 1

CASE_REGEX = r"^as_"
CONTROL_REGEX = r"^(?:h_|hd_)"

COMPUTE_PGEN_IF_MISSING = False
PGEN_PROCESSES = 16

TOPSIS_METRICS = {
    "sample_fraction": "benefit",
    "as_usage": "benefit",
    "h_usage": "cost",
    "log_pgen": "cost",
    "patient_fc": "benefit",
    "log_cluster_size": "benefit",
}

TOPSIS_WEIGHTS = {
    "sample_fraction": 1.5,
    "as_usage": 2.0,
    "h_usage": 1.25,
    "log_pgen": 1.5,
    "patient_fc": 2.0,
    "log_cluster_size": 1.0,
}

MIN_SAMPLES_FOR_RANKING = 2
TOP_N_LOGOS = 12


In [ ]:
runs = load_many_runs(RUNS_ROOT, SAMPLE_NAMES)
matcher = build_reference_matcher(AS_SEQS)
print(f"requested runs: {len(SAMPLE_NAMES)}")
print(f"loaded runs: {len(runs)}")
list(runs)[:5]


In [ ]:
volcano_fig, volcano_summaries = plot_sample_volcano_grid(
    runs,
    matcher,
    fold_threshold=1.0,
    pval_threshold=0.05,
    as_match_threshold=AS_MATCH_THRESHOLD,
)
plt.show()


In [ ]:
merged_df = prepare_merged_clonotypes(
    runs,
    matcher,
    compute_pgen_if_missing=COMPUTE_PGEN_IF_MISSING,
    pgen_processes=PGEN_PROCESSES,
    as_match_threshold=AS_MATCH_THRESHOLD,
    case_regex=CASE_REGEX,
    control_regex=CONTROL_REGEX,
)

merged_df, merge_mapping = merge_clusters_by_cdr3_threshold(
    merged_df,
    substitutions=MERGE_SUBSTITUTIONS,
    cluster_col="cluster_uid",
    seq_col="cdr3aa_beta",
    merged_col="merged_cluster_id",
)

summary = summarize_merged_clusters(merged_df)
summary.head()


In [ ]:
ranked_df = rank_clusters_with_topsis(
    summary,
    min_samples=MIN_SAMPLES_FOR_RANKING,
    metric_types=TOPSIS_METRICS,
    weights=TOPSIS_WEIGHTS,
)

ranked_df[[
    "rank",
    "merged_cluster_id",
    "as_pattern",
    "cluster_size",
    "n_samples",
    "sample",
    "background",
    "sample_fraction",
    "as_usage",
    "h_usage",
    "fisher_p",
    "log_pgen",
    "patient_fc",
    "topsis_score",
]].head(30)


In [ ]:
rank_fig = plot_ranked_metrics(ranked_df)
plt.show()


In [ ]:
logo_fig = plot_top_cluster_logos(
    merged_df,
    ranked_df,
    top_n=TOP_N_LOGOS,
)
plt.show()
